In [1]:

import os
import pandas as pd
import numpy as np
import pywt
from scipy import signal, stats
from matplotlib import font_manager, rcParams
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
import pathlib
from skimage.restoration import denoise_wavelet

# 设置中文字体（可选）
font_path = "../wqy-zenhei.ttc"
font_manager.fontManager.addfont(font_path)
font_prop = font_manager.FontProperties(fname=font_path)
rcParams['font.sans-serif'] = [font_prop.get_name()]
rcParams['axes.unicode_minus'] = False

In [2]:


# 采样参数
FS = 25600

# %%
def compute_psd(x, fs=FS, nperseg=None):
    if nperseg is None:
        nperseg = min(len(x), 4096)
    freqs, psd = signal.welch(x - np.mean(x), fs=fs, window='hann', nperseg=nperseg)
    return freqs, psd

def extract_time_features(x):
    rms = np.sqrt(np.mean(x**2))
    peak = np.max(np.abs(x))
    return {
        "rms": rms,
        "std": np.std(x),
        "mean": np.mean(x),
        "kurtosis": stats.kurtosis(x),
        "skewness": stats.skew(x),
        "peak2peak": np.ptp(x),
        "crest_factor": peak / (rms + 1e-12),
        "impulse_factor": peak / (np.mean(np.abs(x)) + 1e-12),
        "clearance_factor": peak / ((np.mean(np.sqrt(np.abs(x)))**2) + 1e-12),
    }

def extract_psd_features(x, fs=FS, n_bins=10):
    freqs, psd = compute_psd(x, fs)
    psd_energy = psd.sum() + 1e-12
    psd_norm = psd / psd_energy
    centroid = np.sum(freqs * psd_norm)
    bandwidth = np.sqrt(np.sum(((freqs - centroid)**2) * psd_norm))
    entropy = -np.sum(psd_norm * np.log(psd_norm + 1e-12))
    flatness = np.exp(np.mean(np.log(psd + 1e-12))) / (np.mean(psd) + 1e-12)
    feats = {
        "psd_energy": psd_energy,
        "spectral_entropy": entropy,
        "spectral_centroid": centroid,
        "spectral_bandwidth": bandwidth,
        "spectral_flatness": flatness
    }
    # 分频能量
    fmax = freqs[-1]
    bins = np.linspace(0, fmax, n_bins+1)
    for i in range(n_bins):
        idx = (freqs >= bins[i]) & (freqs < bins[i+1])
        feats[f"psd_bin_energy_{i}"] = psd[idx].sum() / psd_energy
    return feats

def extract_envelope_features(x, fs=FS, n_bins=8):
    analytic = signal.hilbert(x - np.mean(x))
    env = np.abs(analytic)
    env -= env.mean()
    freqs, psd_env = compute_psd(env, fs)
    psd_env += 1e-12
    psd_env_norm = psd_env / psd_env.sum()
    centroid = np.sum(freqs * psd_env_norm)
    entropy = -np.sum(psd_env_norm * np.log(psd_env_norm + 1e-12))
    feats = {
        "env_rms": np.sqrt(np.mean(env**2)),
        "env_kurtosis": stats.kurtosis(env),
        "env_entropy": entropy,
        "env_centroid": centroid
    }
    fmax = freqs[-1]
    bins = np.linspace(0, fmax, n_bins+1)
    for i in range(n_bins):
        idx = (freqs >= bins[i]) & (freqs < bins[i+1])
        feats[f"env_bin_energy_{i}"] = psd_env[idx].sum() / psd_env.sum()
    return feats

def extract_wavelet_features(x):
    x = x - np.mean(x)
    coeffs = pywt.wavedec(x, 'db4', level=4)
    energies = np.array([np.sum(c**2) for c in coeffs])
    total_energy = energies.sum() + 1e-12
    high_ratio = (energies[-1] + energies[-2]) / total_energy
    energy_norm = energies / total_energy
    wavelet_entropy = -np.sum(energy_norm * np.log(energy_norm + 1e-12))
    return {
        "wavelet_energy": total_energy,
        "wavelet_high_ratio": high_ratio,
        "wavelet_entropy": wavelet_entropy
    }

def extract_features(x, fs=FS):
    # x = denoise_wavelet(x, method='BayesShrink', mode='soft', wavelet='db4', rescale_sigma=True)

    feats = {}
    feats.update(extract_time_features(x))
    feats.update(extract_psd_features(x, fs))
    feats.update(extract_envelope_features(x, fs))
    feats.update(extract_wavelet_features(x))
    return feats

In [3]:
def process_bearing_folder(folder, delta_t=10):
    acc_files = sorted([os.path.join(folder, f) for f in os.listdir(folder)
                        if f.startswith("acc") and f.endswith(".csv")])
    
    feats_list = []
    for file in acc_files:
        try:
            df = pd.read_csv(file, header=None, sep=',').to_numpy()
            sig = np.sqrt(df[:, 4]**2 + df[:, 5]**2)
        except:
            df = pd.read_csv(file, header=None, sep=';').to_numpy()
            sig = np.sqrt(df[:, 4]**2 + df[:, 5]**2)
        # 对水平和垂直进行均方根
        
        # try:
        #     sig = pd.read_csv(file, header=None, sep=',').to_numpy()[:,4]
        # except:
        #     sig = pd.read_csv(file, header=None, sep=';').to_numpy()[:,4]
        # TODO：这里可以考虑增加一些数据清洗步骤，比如去除异常值、平滑信号等
        feats_list.append(extract_features(sig))
    df_feats = pd.DataFrame(feats_list)
    
    # 1. 去掉大幅提升效果
    # df_smooth = df_feats.ewm(alpha=0.3).mean()
    # df_smooth.columns = [f"{c}_smooth" for c in df_smooth.columns]
    # df_feats = pd.concat([df_feats, df_smooth], axis=1)
   
    # 2. RUL 标签（真实秒数，不泄漏未来信息）
    rul = np.arange(len(acc_files)-1, -1, -1) * delta_t
    df_feats['RUL'] = rul
    
    df_feats['life'] = np.arange(len(acc_files)) * delta_t
    # 3. 
    sub = pathlib.Path(folder).stem
    df_feats['bearing_id'] = sub
    # 4. 增加bearing type反而效果会下降
    # if sub.startswith("Bearing1_"):
    #     df_feats['BearType'] = 1
    # elif sub.startswith("Bearing2_"):
    #     df_feats['BearType'] = 2
    # elif sub.startswith("Bearing3_"):
    #     df_feats['BearType'] = 3
    # else:
    #     raise ValueError(f"Unexpected bearing name: {sub}")
    
    return df_feats

def build_train_dataset(data_path, window_scales=[3, 5], step_size=5):
    """
    短期: 2~5个采样点， 对应20~50秒
    中期：
    长期：
    """
    all_dfs = []
    print(f"Processing training data with scales: {window_scales}...")
    
    # 1. 基础特征提取
    for sub in os.listdir(data_path):
        folder = os.path.join(data_path, sub)
        if not os.path.isdir(folder): continue
        print(f"  -> {sub}")
        df = process_bearing_folder(folder) # 使用你原有的基础特征提取函数
        all_dfs.append(df)

    full_raw_df = pd.concat(all_dfs).reset_index(drop=True)
    
    # 确定哪些列需要进行滚动计算（排除标签和ID）
    base_feature_cols = [c for c in full_raw_df.columns if c not in ['RUL', 'bearing_id']]
    
    processed_X = []
    processed_y = []
    processed_groups = []
    
    # 2. 按轴承分组进行多尺度特征工程
    for bid, group in full_raw_df.groupby('bearing_id'):
        group = group.reset_index(drop=True)
        
        # 初始特征池
        feat_pool = [group[base_feature_cols]]
        
        raw_roll_means = {}
        
        for w in window_scales:
            # --- 均值特征：滤除高频噪声，提取平稳退化趋势 ---
            roll_mean = group[base_feature_cols].rolling(window=w, min_periods=1).mean()
            roll_mean.columns = [f"{c}_w{w}_mean" for c in base_feature_cols]
            raw_roll_means[w] = roll_mean
            
            # --- 稳定性特征：轴承损坏时，振动幅值的波动会剧增 ---
            roll_std = group[base_feature_cols].rolling(window=w, min_periods=1).std().fillna(0)
            roll_std.columns = [f"{c}_w{w}_std" for c in base_feature_cols]
            
            # --- 趋势特征（Delta）：当前均值与 N 个时刻前的均值之差，捕捉退化斜率 ---
            # 使用 w//2 作为偏移量，捕捉中短期变化
            #roll_delta = roll_mean - roll_mean.shift(w // 2).bfill()
            roll_delta = roll_mean.diff(w // 2).fillna(0)
            roll_delta.columns = [f"{c}_w{w}_delta" for c in base_feature_cols]
            
            feat_pool.extend([roll_mean, roll_std, roll_delta])
            
        
        # 为每个窗口尺度计算差分特征
        for i in range(len(window_scales)-1):
            w_s, w_l = window_scales[i], window_scales[i+1]
            
  
            diff = pd.DataFrame(
                raw_roll_means[w_s].values - raw_roll_means[w_l].values, 
                columns=[
                f"{col}_gap_w{w_s}_w{w_l}"
                for col in raw_roll_means[w_s].columns
            ])
            feat_pool.append(diff)      
        
        # 合并所有尺度特征
        X_group = pd.concat(feat_pool, axis=1)
        y_group = group['RUL']
        
        # 3. 筛选退化阶段（可选优化）
        # 轴承前期通常非常稳定，RUL预测意义不大。我们可以切掉前 20% 的健康数据，
        # 让模型更专注于捕捉退化期的细微变化。
        n = len(X_group)
        start_idx = int(n * 0.2) 
        
        # 4. 步进采样（Sampling）
        # 我们不需要把每一秒的数据都喂给模型（太冗余），每隔 step_size 取一个点
        indices = np.arange(start_idx, n, step_size)
        
        processed_X.append(X_group.iloc[indices])
        processed_y.append(np.log1p(y_group.iloc[indices])) # RUL Log变换
        processed_groups.extend([bid] * len(indices))

    X = pd.concat(processed_X, axis=0).reset_index(drop=True)
    y = pd.concat(processed_y, axis=0).reset_index(drop=True)
    groups = np.array(processed_groups)
    
    print(f"Final Dataset Shape: {X.shape}")
    return X, y, groups
base_path = "./phm-ieee-2012-data-challenge-dataset-master/Learning_set"
X, y, groups = build_train_dataset(base_path)
# LightGBM 参数

params = {
    'learning_rate': 0.02,
          'boosting_type': 'gbdt',
          'objective': 'regression', 
          'metric': 'mae',
          'num_leaves': 67, 
          'verbose': -1,
          'seed': 2222, 
          'n_jobs': 32,
          'min_child_weight': 9, 
          'max_depth': 6,
          'lambda_l1': 0.8,
          'lambda_l2': 1.5,
          'feature_fraction': 0.5, 
          'bagging_fraction': 0.95, 
          'bagging_freq': 5, 
          }


gkf = GroupKFold(n_splits=5)
models = []
oof_preds = np.zeros(len(X))
scores = []

for fold, (trn_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups)):
    X_train, y_train = X.iloc[trn_idx], y.iloc[trn_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    print(f"Fold {fold+1} Train on: {np.unique(groups[trn_idx])} Validating on: {np.unique(groups[val_idx])}")
    lgb_train = lgb.Dataset(X_train, y_train)
    lgb_val = lgb.Dataset(X_val, y_val)
    
    model = lgb.train(params, train_set=lgb_train,
                      valid_sets=[lgb_train, lgb_val],
                      valid_names=['train', 'valid'],
                      num_boost_round=3000,
                      callbacks=[lgb.early_stopping(50, first_metric_only=True), lgb.log_evaluation(0)])
    
    val_pred = np.expm1(model.predict(X_val))
    y_val_true = np.expm1(y_val)
    
    score = mean_absolute_error(y_val_true, val_pred)
    
    oof_preds[val_idx] = val_pred
    
    scores.append(score)
    models.append(model)
    print(f"Fold {fold+1} MAE: {score:.2f}")
print(f"Average MAE: {np.mean(scores):.2f}")


Processing training data with scales: [3, 5]...
  -> Bearing1_2
  -> Bearing3_2
  -> Bearing1_1
  -> Bearing2_1
  -> Bearing2_2
  -> Bearing3_1
Final Dataset Shape: (1208, 320)
Fold 1 Train on: ['Bearing1_2' 'Bearing2_1' 'Bearing2_2' 'Bearing3_1' 'Bearing3_2'] Validating on: ['Bearing1_1']
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	train's l1: 0.778697	valid's l1: 1.28336
Evaluated only: l1
Fold 1 MAE: 8633.10
Fold 2 Train on: ['Bearing1_1' 'Bearing1_2' 'Bearing2_1' 'Bearing2_2' 'Bearing3_1'] Validating on: ['Bearing3_2']
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	train's l1: 0.887071	valid's l1: 0.779887
Evaluated only: l1
Fold 2 MAE: 3736.51
Fold 3 Train on: ['Bearing1_1' 'Bearing1_2' 'Bearing2_2' 'Bearing3_1' 'Bearing3_2'] Validating on: ['Bearing2_1']
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[47]	train's l1: 0.43002

In [4]:
# import shap, tqdm

# def view_top_features():
#     for index, model in tqdm.tqdm(enumerate(models)):
#         explainer = shap.TreeExplainer(model)
#         shap_values = explainer.shap_values(X)
#         feature_importance = np.abs(shap_values).mean(axis=0)
#         feat_imp_df = pd.DataFrame({
#             'feature': X.columns,
#             'importance': feature_importance
#         }).sort_values(by='importance', ascending=False)
        
#         top100 = feat_imp_df.head(100)
#         plt.figure(figsize=(18, 30)) 
#         plt.barh(top100['feature'], top100['importance'])
        
#         plt.title(f'Fold {index} Feature Importance')
#         plt.xlabel("Importance")
#         plt.ylabel("Feature")
        
#         plt.gca().invert_yaxis()  # 让重要性高的在上方
#         plt.tight_layout()
#         plt.show()
# view_top_features()

In [5]:
import shap


from collections import defaultdict

feature_importance_score = defaultdict(list)
for model in models:
    feature_names = model.feature_name()
    # importances = model.feature_importance(importance_type='gain')
    # for name, imp in zip(feature_names, importances):
    #     feature_importance_score[name].append(imp)
    
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)
    importance = np.abs(shap_values).mean(axis=0)
    for name, imp in zip(feature_names, importance):
        feature_importance_score[name].append(imp)

feature_agg = {feat: np.median(imps) for feat, imps in feature_importance_score.items()}
feat_imp_df = pd.DataFrame({
    'feature': list(feature_agg.keys()),
    'importance': list(feature_agg.values())
}).sort_values('importance', ascending=False)

top_n = 100
top_features = feat_imp_df.head(top_n)['feature'].tolist()

corr_matrix = X[top_features].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if any(upper[c] >= 0.9)]
top_features = [f for f in top_features if f not in to_drop]
print(len(top_features))
print(top_features)


44
['psd_bin_energy_1_w5_mean', 'rms_w3_mean', 'life', 'wavelet_entropy_w3_mean', 'psd_bin_energy_9_w5_mean', 'env_bin_energy_6_w3_mean', 'psd_bin_energy_3_w5_mean', 'skewness_w5_mean', 'wavelet_energy_w5_mean', 'psd_bin_energy_4_w5_mean', 'env_bin_energy_7_w5_mean', 'psd_bin_energy_2_w5_mean', 'env_bin_energy_1_w5_mean', 'psd_bin_energy_2', 'psd_bin_energy_7_w3_mean', 'psd_bin_energy_3_w5_std', 'psd_bin_energy_4_w5_std', 'env_bin_energy_0_w5_std', 'psd_bin_energy_2_w5_std', 'psd_bin_energy_5_w5_std', 'env_bin_energy_1_w5_std', 'psd_bin_energy_6_w5_std', 'psd_bin_energy_7_w5_std', 'env_bin_energy_2_w5_std', 'psd_bin_energy_8_w5_std', 'env_bin_energy_4_w5_std', 'env_bin_energy_5_w5_std', 'life_w5_std', 'env_bin_energy_6_w5_std', 'env_bin_energy_7_w5_std', 'psd_bin_energy_9_w5_std', 'wavelet_high_ratio_w5_std', 'wavelet_energy_w5_std', 'env_rms_w5_std', 'env_kurtosis_w5_std', 'env_bin_energy_3_w5_std', 'env_entropy_w5_mean', 'wavelet_entropy_w5_std', 'mean_w5_std', 'psd_energy_w5_std', '

In [6]:
X_selected = X[top_features]

final_models = []
scores = []

for fold, (trn_idx, val_idx) in enumerate(gkf.split(X_selected, y, groups)):

    X_train, y_train = X_selected.iloc[trn_idx], y.iloc[trn_idx]
    X_val, y_val = X_selected.iloc[val_idx], y.iloc[val_idx]

    train_set = lgb.Dataset(X_train, y_train)
    val_set = lgb.Dataset(X_val, y_val)

    model = lgb.train(params, train_set=train_set,
                      valid_sets=[train_set, val_set],
                      valid_names=['train', 'valid'],
                      num_boost_round=3000,
                      callbacks=[lgb.early_stopping(50, first_metric_only=True), lgb.log_evaluation(0)])

    val_pred = np.expm1(model.predict(X_val))
    y_val_true = np.expm1(y_val)

    score = mean_absolute_error(y_val_true, val_pred)

    scores.append(score)
    final_models.append(model)

    print(f"[Refit] Fold {fold+1} MAE: {score:.2f}")

print("Final CV MAE:", np.mean(scores))

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	train's l1: 0.780457	valid's l1: 1.28617
Evaluated only: l1
[Refit] Fold 1 MAE: 8641.02
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	train's l1: 0.888267	valid's l1: 0.761129
Evaluated only: l1
[Refit] Fold 2 MAE: 3657.59
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[46]	train's l1: 0.473426	valid's l1: 0.696702
Evaluated only: l1
[Refit] Fold 3 MAE: 1911.46
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[35]	train's l1: 0.525075	valid's l1: 0.612762
Evaluated only: l1
[Refit] Fold 4 MAE: 1519.19
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[16]	train's l1: 0.651235	valid's l1: 0.941935
Evaluated only: l1
[Refit] Fold 5 MAE: 2382.53
Final CV MAE: 3622.359934840953


In [7]:
for index, model in enumerate(final_models):
    model.save_model(f'm{index}.txt')

In [8]:

def predict_test_folder(folder, models, top_features, window_scales=[3, 5]):
    """
    匹配多尺度特征逻辑的测试集预测函数
    """
    # 1. 基础特征提取（每个文件提取一行特征）
    df_raw = process_bearing_folder(folder)
    base_feature_cols = [c for c in df_raw.columns if c not in ['RUL', 'bearing_id']]
    
    # 2. 复刻多尺度特征池逻辑
    feat_pool = [df_raw[base_feature_cols]]
    
    raw_roll_means = {}

    for w in window_scales:
        # 均值
        roll_mean = df_raw[base_feature_cols].rolling(window=w, min_periods=1).mean()
        roll_mean.columns = [f"{c}_w{w}_mean" for c in base_feature_cols]
        
        raw_roll_means[w]  = roll_mean   
        
        # 稳定性 (标准差)
        roll_std = df_raw[base_feature_cols].rolling(window=w, min_periods=1).std().fillna(0)
        roll_std.columns = [f"{c}_w{w}_std" for c in base_feature_cols]
        
        # 趋势 (Delta)
        roll_delta = roll_mean.diff(w // 2).fillna(0)
        roll_delta.columns = [f"{c}_w{w}_delta" for c in base_feature_cols]
        
        feat_pool.extend([roll_mean, roll_std, roll_delta])
    
    # 为每个窗口尺度计算差分特征
    for i in range(len(window_scales)-1):
        w_s, w_l = window_scales[i], window_scales[i+1]
                    
        diff = pd.DataFrame(
            raw_roll_means[w_s].values - raw_roll_means[w_l].values, 
            columns=[
            f"{col}_gap_w{w_s}_w{w_l}"
            for col in raw_roll_means[w_s].columns
        ])
   
        feat_pool.append(diff)     
    # 合并生成完整的特征矩阵
    X_full = pd.concat(feat_pool, axis=1)
    
    # 3. 稳健预测策略：取最后 N 个时刻的平均预测值
    # 轴承退化在末期可能波动较大，取最后 3 个点的预测均值可以有效过滤瞬时异常
    X_test_final = X_full.tail(3)[top_features] 
    
    all_fold_preds = []
    for index, model in enumerate(models):
        disk_model = lgb.Booster(model_file=f'm{index}.txt')
        # 模型输出的是 log1p(RUL)，需要反对数变换
        p_log = model.predict(X_test_final)
        
        disk_p_log = disk_model.predict(X_test_final)
        assert np.allclose(p_log, disk_p_log)
        p_real = np.expm1(p_log)
        # print('-->',folder,p_real)
        all_fold_preds.append(np.mean(p_real))
    
    # 返回所有 Fold 模型的集成预测值
    final_pred = np.mean(all_fold_preds)
    
    # 物理约束：RUL 不能小于 0
    return max(0.0, final_pred)

    

# 测试集预测
test_base = "./phm-ieee-2012-data-challenge-dataset-master/Full_Test_Set"
# 真实标签
test_rul_map = {
    "Bearing1_3": 5730,
    "Bearing1_4": 339,
    "Bearing1_5": 1610,
    "Bearing1_6": 1460,
    "Bearing1_7": 7570,
    "Bearing2_3": 7530,
    "Bearing2_4": 1390,
    "Bearing2_5": 3090,
    "Bearing2_6": 1290,
    "Bearing2_7": 580,
    "Bearing3_3": 820,
}

test_rul_pred_map = {}
for sub in os.listdir(test_base):
    folder = os.path.join(test_base, sub)
    if not os.path.isdir(folder): continue
    pred_rul = predict_test_folder(folder, final_models, top_features=top_features)
    test_rul_pred_map[sub] = pred_rul
    print(f"{sub}: Predicted RUL = {pred_rul:.2f} seconds, True RUL = {test_rul_map[sub]} seconds")

total_error = 0
for bearing, y_true in test_rul_map.items():
    y_pred = test_rul_pred_map[bearing]
    total_error += abs(float(y_true) - y_pred)
    
print(f'{total_error=}')


Bearing3_3: Predicted RUL = 2183.93 seconds, True RUL = 820 seconds
Bearing1_3: Predicted RUL = 2235.93 seconds, True RUL = 5730 seconds
Bearing2_6: Predicted RUL = 2167.87 seconds, True RUL = 1290 seconds
Bearing1_7: Predicted RUL = 2185.09 seconds, True RUL = 7570 seconds
Bearing2_4: Predicted RUL = 2609.36 seconds, True RUL = 1390 seconds
Bearing1_4: Predicted RUL = 2143.15 seconds, True RUL = 339 seconds
Bearing2_7: Predicted RUL = 2606.82 seconds, True RUL = 580 seconds
Bearing2_3: Predicted RUL = 2632.83 seconds, True RUL = 7530 seconds
Bearing2_5: Predicted RUL = 2839.47 seconds, True RUL = 3090 seconds
Bearing1_5: Predicted RUL = 2178.71 seconds, True RUL = 1610 seconds
Bearing1_6: Predicted RUL = 2203.80 seconds, True RUL = 1460 seconds
total_error=np.float64(22631.320628413345)


结论: 这里rolling太多会降低对时间变化的敏感度，不如baseline1